# ML Model - Q-Score PredictionConverted from `src/ml_model_q.py`---

**Beschreibung:** ML Model - Trains an ensemble model for score prediction

In [1]:
import pandas as pdimport numpy as npfrom pathlib import Pathimport joblibfrom sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import (    accuracy_score, cohen_kappa_score, mean_absolute_error,    confusion_matrix, f1_score)from sklearn.ensemble import RandomForestClassifier, VotingClassifierdef quadratic_weighted_kappa(y_true, y_pred, num_classes=5):    """    Calculate Quadratic Weighted Kappa (QWK).    Penalizes larger deviations more than smaller ones.    """    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))    weights = np.zeros((num_classes, num_classes))    for i in range(num_classes):        for j in range(num_classes):            weights[i, j] = ((i - j) ** 2) / ((num_classes - 1) ** 2)    hist_true = np.bincount(y_true, minlength=num_classes)    hist_pred = np.bincount(y_pred, minlength=num_classes)    n = len(y_true)    expected = np.outer(hist_true, hist_pred).astype(float) / n    num = np.sum(weights * cm)    den = np.sum(weights * expected)    if den == 0:        return 1.0    return 1.0 - (num / den)# Optional: XGBoost and LightGBMtry:    import xgboost as xgb    XGBOOST_AVAILABLE = Trueexcept ImportError:    XGBOOST_AVAILABLE = Falsetry:    import lightgbm as lgb    LIGHTGBM_AVAILABLE = Trueexcept ImportError:    LIGHTGBM_AVAILABLE = Falsedef create_model():    """    Create ML ensemble.    Consists of:    - XGBoost (if available)    - LightGBM (if available)    - RandomForest (always available)    """    estimators = []    # RandomForest (base)    rf = RandomForestClassifier(        n_estimators=100,        max_depth=6,  # Optimized for balance between bias and variance        random_state=42,        n_jobs=-1    )    estimators.append(('rf', rf))    # XGBoost    if XGBOOST_AVAILABLE:        xgb_model = xgb.XGBClassifier(            n_estimators=100,            max_depth=6,  # Consistent with other models            learning_rate=0.1,            random_state=42,            verbosity=0        )        estimators.append(('xgb', xgb_model))    # LightGBM    if LIGHTGBM_AVAILABLE:        lgb_model = lgb.LGBMClassifier(            n_estimators=100,            max_depth=6,  # Consistent with other models            learning_rate=0.1,            random_state=42,            verbose=-1        )        estimators.append(('lgb', lgb_model))    # Voting Ensemble    ensemble = VotingClassifier(        estimators=estimators,        voting='soft'    )    return ensembledef train_model(X, y, test_size=0.2):    """    Train the model.    Args:        X: Features DataFrame        y: Target DataFrame (Q1, Q2, Q3)        test_size: Test data proportion    Returns:        dict: Trained models and metrics    """    print("\n MODEL TRAINING")    print("="*50)    targets = ['Q1', 'Q2', 'Q3']    models = {}    metrics = {}    feature_importance = {}    # Scaler    scaler = StandardScaler()    for target in targets:        print(f"\n Training for {target}...")        # Prepare target (Scores 1-5 -> 0-4)        y_target = y[target].values - 1        # Train-Test Split        X_train, X_test, y_train, y_test = train_test_split(            X.values, y_target,            test_size=test_size,            random_state=42,            stratify=y_target        )        # Scaling        X_train_scaled = scaler.fit_transform(X_train)        X_test_scaled = scaler.transform(X_test)        print(f"   Train: {len(X_train)}, Test: {len(X_test)}")        # Create and train model        model = create_model()        model.fit(X_train_scaled, y_train)        # Prediction        y_pred = model.predict(X_test_scaled)        # Base metrics        accuracy = accuracy_score(y_test, y_pred)        mae = mean_absolute_error(y_test, y_pred)        # F1-Scores        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)        # Kappa metrics        kappa = cohen_kappa_score(y_test, y_pred)        qwk = quadratic_weighted_kappa(np.array(y_test), np.array(y_pred))        # Cross-Validation        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)        cv_scores = cross_val_score(model, scaler.fit_transform(X.values), y_target, cv=cv, scoring='accuracy')        metrics[target] = {            'accuracy': round(accuracy, 3),            'mae': round(mae, 3),            'f1_macro': round(f1_macro, 3),            'f1_weighted': round(f1_weighted, 3),            'kappa': round(kappa, 3),            'qwk': round(qwk, 3),            'cv_mean': round(cv_scores.mean(), 3),            'cv_std': round(cv_scores.std(), 3),            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()        }        print(f"   Accuracy: {accuracy:.3f}")        print(f"   MAE: {mae:.3f}")        print(f"   Macro-F1: {f1_macro:.3f}")        print(f"   Weighted-F1: {f1_weighted:.3f}")        print(f"   Kappa: {kappa:.3f}")        print(f"   QWK: {qwk:.3f}")        print(f"   CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")        models[target] = model        # Feature Importance (from RandomForest)        if hasattr(model.named_estimators_['rf'], 'feature_importances_'):            importance_df = pd.DataFrame({                'feature': X.columns,                'importance': model.named_estimators_['rf'].feature_importances_            }).sort_values('importance', ascending=False)            feature_importance[target] = importance_df    return {        'models': models,        'scaler': scaler,        'metrics': metrics,        'feature_importance': feature_importance    }def save_model(model_data, output_path="models/q_score_model.joblib"):    """Save the trained model."""    output_path = Path(output_path)    output_path.parent.mkdir(parents=True, exist_ok=True)    joblib.dump(model_data, output_path)    print(f"\n Model saved: {output_path}")def load_model(model_path="models/q_score_model.joblib"):    """Load a saved model."""    model_path = Path(model_path)    if model_path.exists():        return joblib.load(model_path)    return Nonedef print_summary(metrics):    """Print summary."""    print("\n" + "="*50)    print(" MODEL SUMMARY")    print("="*50)    for target, m in metrics.items():        print(f"\n{target}:")        print(f"   Accuracy: {m['accuracy']}")        print(f"   Kappa: {m['kappa']}")        print(f"   CV: {m['cv_mean']} ± {m['cv_std']}")    # Average    avg_acc = np.mean([m['accuracy'] for m in metrics.values()])    avg_kappa = np.mean([m['kappa'] for m in metrics.values()])    print(f"\n TOTAL:")    print(f"   Avg Accuracy: {avg_acc:.3f}")    print(f"   Avg Kappa: {avg_kappa:.3f}")

##  Execution

In [2]:
print("="*50)print(" ML MODEL TRAINING")print("="*50)# Load ML datasetdata_path = Path("data/processed/ml_dataset.csv")if data_path.exists():    df = pd.read_csv(data_path)    print(f" Loaded: {len(df)} samples")    # Separate features and targets    target_cols = ['Q1', 'Q2', 'Q3']    feature_cols = [col for col in df.columns if col not in target_cols]    X = df[feature_cols]    y = df[target_cols]    print(f"   Features: {len(feature_cols)}")    # Training    model_data = train_model(X, y)    # Summary    print_summary(model_data['metrics'])    # Save    save_model(model_data)    # Show top features    print("\n Top 5 Features (Q1):")    if 'Q1' in model_data['feature_importance']:        for _, row in model_data['feature_importance']['Q1'].head(5).iterrows():            print(f"   {row['feature']}: {row['importance']:.4f}")else:    print(" ML dataset not found!")    print("   Please run feature_engineering.py first.")

================================================== ML MODEL TRAINING================================================== ML dataset not found!   Please run feature_engineering.py first.